## DATABRICKS CAPSTONE PROJECTS
<br>**DATE: 2026-09-02**
<br>**TOPIC: Customer Transformation in Silver Layer**

<br>**Before transformation- apply the SCD load for UPSERT**


| # | Transformation Task |
|---|----------------------|
| 1 | Remove duplicates | 
| 2 | Keep the latest record |
| 3 | Trim whitespace | 
| 4 | Convert email to lowercase | 
| 5 | Validate email format | 
| 6 | Handle missing email/phone |
| 7 | Standardize city/state |
| 8 | Validate customer status |

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
try:

    df_customer = spark.table("ecommerce.bronze.customers")
    count = df_customer.count()
    print(f"Bronze customer table loaded with {count} rows")
    
except Exception as e:
    raise e

In [0]:
try:

    df_customer_dedup = df_customer.dropDuplicates()
    count = df_customer_dedup.count()
    print(f"Bronze customer table post deduplication has {count} rows")
    
except Exception as e:
    raise e

In [0]:
try:
    # get list of all string columns--REMOVE WHITE SPACE
    string_cols = [k for k, v in df_customer_dedup.dtypes if (v) == "string"]

    for i in string_cols:
        df_customer_dedup = df_customer_dedup.withColumn(i, trim(i))

except Exception as e:
    raise e

In [0]:
try:
    df_customer_dedup = df_customer_dedup.withColumn("email", lower(col("email")))

except Exception as e:
    raise e

In [0]:
try:
    cols = ["email", "phone"]
    for i in cols:
        df_customer_dedup = df_customer_dedup.withColumn(
            i, when(col(i).isNull(), lit("NA")).otherwise(col(i))
        )


except Exception as e:
    raise e

In [0]:
try:
    cols = ['city','state','customer_status']
    for i in cols:
        df_customer_dedup = df_customer_dedup.withColumn(i,initcap(col(i)))

except Exception as e:
    raise e

In [0]:
try:
    valid_status = ["Active", "Inactive"]
    df_customer_dedup = df_customer_dedup.withColumn(
        "customer_status",
        when(
            col("customer_status").isin(valid_status), col("customer_status")
        ).otherwise(None),
    )

except Exception as e:
    raise e

In [0]:
try:
    email_pattern = r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"

    df_customer_dedup = df_customer_dedup.withColumn(
        "email",
        when(col("email").rlike(email_pattern), col("email")).otherwise(
            lit("Invalid Email")
        ),
    )

except Exception as e:
    raise e

In [0]:
try:
    window_spec = Window.partitionBy("customer_id").orderBy(desc("updated_at"))

    df_customer_dedup = df_customer_dedup.withColumn(
        "Seq", row_number().over(window_spec)
    )

    df_customer_final = df_customer_dedup.filter(col("Seq") == 1).drop("Seq")


except Exception as e:
    raise e

In [0]:
try:
    df_customer_final.write.saveAsTable('ecommerce.silver.customers', mode = "OVERWRITE")
except Exception as e:
    raise e